# 柔性生产线：实验结果复核

先在项目目录运行 `uv run soft-procedure experiments`，再用 Python 内核执行下列单元。保留原始 `soft-procedure.ipynb`，本分析本单独读取实际运行结果。

In [ ]:
from pathlib import Path
import json
import csv
from statistics import mean, stdev
from math import sqrt

root = Path.cwd()
suite = json.loads((root / "outputs/results.json").read_text())
print("观察时长:", suite["base_config"]["horizon"], "min")
print("种子:", suite["seeds"])
for s in suite["scenarios"]:
    print(s["name"], s["summary"]["completed"], s["paired_output_difference"])

## 配对差值

先按种子逐轮相减，再计算 t 区间；不把两组独立均值区间当成差值区间。

In [ ]:
from soft_procedure.experiments import estimate
base = suite["scenarios"][0]
for s in suite["scenarios"][1:]:
    diffs = [a["completed"] - b["completed"] for a,b in zip(s["runs"],base["runs"])]
    print(s["name"], estimate(diffs))
    assert estimate(diffs) == s["paired_output_difference"]

## 核对逐轮工件守恒

成品、线内在制和线外待投料的统计口径必须一致。

In [ ]:
with (root / "outputs/replications.csv").open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f))
for row in rows:
    assert int(row["arrived"]) == int(row["completed"]) + int(row["wip"]) + int(row["external"])
print(f"{len(rows)} 轮守恒检查通过。")

## 交互查看

下列嵌入只展示已有结果。使用 `uv run soft-procedure serve` 才能修改参数并重新计算。

In [ ]:
from IPython.display import IFrame, display
display(IFrame("outputs/dashboard.html", width="100%", height=900))